# LipPY AV-Fusion v3 — Co-train + Symmetric Fusion + Hybrid CTC/Attention

**v3 = AV-HuBERT / Auto-AVSR mimari prensipleri:**

v2'de: 2 encoder donmus + 3-6 layer cross-attn + sadece CTC → PER %33 plateau (Auto-AVSR %0.9, biz %33).

v3'te 3 buyuk fark:

| | v2 | **v3** |
|---|---|---|
| Video encoder | DONMUS (v1c) | **Ust 2 Conformer layer TRAINABLE** (v1cc) |
| Audio encoder | DONMUS (Whisper medium) | **Ust 4 Whisper layer TRAINABLE** |
| Fusion | Asymmetric cross-attn (V=Q, A=K/V) | **SYMMETRIC concat + 8-layer shared Transformer** |
| Modality dropout | Yok | **0.5** (AV-HuBERT-tarzi) |
| Head | Sadece CTC | **Hybrid: 0.1*CTC + 0.9*Attention seq2seq** |
| LR strategy | Tek LR | **3-grup: video_top=1e-5, audio_top=1e-5, fusion+head=5e-4** |
| Trainable | 5-10M | **~75M** (78M = 50 audio + 8 video + 20 fusion+head) |

## Beklenen PER

v1 medium audio %31 + v1c video %50 + v2 fusion → %33 plateau
v3 PLAN A.2 medium %27-30 + v1cc %48 + co-train + hybrid → **%14-22 hedef**

## Inputs (Kaggle)

- `brokyewing/lippy-test` (ROI + text + splits)
- `brokyewing/lippy-audio-segments` (audio .wav)
- `brokyewing/lippy-phoneme-v1cc-full-checkpoint` (v1cc video encoder, PER %48)
- `brokyewing/lippy-audio-whisper-ctc-v2-checkpoint` (PLAN A.2 medium audio, hedef PER %27-30)
- `brokyewing/lippy-mel-cache-whisper-medium` (80-bin mel cache)
- (opsiyonel) `brokyewing/lippy-av-fusion-v3-checkpoint` (resume)

## Sure

- Hybrid head decoder forward+backward +%20-30 compute vs v2
- Epoch tahmini: 3-4h (medium + 10 epoch + ~75M trainable)
- 10 epoch toplam: 30-40h (3-4 Kaggle session, ~3-4 gun calendar)

In [ ]:
!pip install -q transformers torchaudio soundfile

import os, glob, json, shutil, time, math, random, subprocess
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torchaudio.models import Conformer
from transformers import WhisperModel, WhisperFeatureExtractor

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'CPU')

# ============ CONFIG (v3.2: NaN-safe LR + flip alpha) ============
# v3.1 sonucu: NaN 13331/13336 batch -> %99.96 gradient explosion
# Sebep: LR_FUSION_HEAD 5e-4 (sifirdan attention decoder icin cok yuksek)
#        + CTC_ATTN_ALPHA 0.1 (%90 attention weight, sifirdan decoder = NaN)
# v3.2 fix: LR 5e-4 -> 1e-4, ALPHA 0.1 -> 0.7 (CTC oncelikli),
#           GCLIP 2.0 -> 1.0 (sıkı clip), WARMUP 1 -> 2 (smooth ramp)
EPOCHS = 10
GCLIP = 1.0                             # v3.2: 2.0 -> 1.0 (NaN clip sıkı)
ACCUM = 4
BS = 4                                  # effective 16
UPLOAD_EVERY = 2

# 3-grup LR (Auto-AVSR pattern -- ama bizimkiler scratch, daha dusuk LR)
LR_VIDEO_TOP = 1e-5                     # v1cc top 2 conformer layer
LR_AUDIO_TOP = 1e-5                     # Whisper top 4 layer
LR_FUSION_HEAD = 1e-4                   # v3.2: 5e-4 -> 1e-4 (NaN-safe)
WARMUP = 2                              # v3.2: 1 -> 2 (smooth ramp ep1=0.5x peak)

# Fusion mimari
D_MODEL = 512
NHEAD = 8
N_FUSION_LAYERS = 8                     # shared Transformer
N_DECODER_LAYERS = 4                    # attention decoder
FUSION_DROPOUT = 0.1
MODALITY_DROPOUT = 0.5                  # AV-HuBERT-tarzi (train only)

# Hybrid loss
# v3.2: alpha 0.1 -> 0.7 (CTC oncelikli, attention destek)
# Sebep: Auto-AVSR alpha=0.1 kullanır AMA encoder+decoder hepsi pretrained.
# Bizde decoder sifirdan -> attention loss patlar. CTC stable, ona yaslan.
CTC_ATTN_ALPHA = 0.7                    # v3.2: 0.1 -> 0.7 (0.7*CTC + 0.3*Attn)

# Unfreeze
UNFREEZE_AUDIO_TOP = 4                  # Whisper-medium 24 layer, top 4 trainable (ust 20 donmus)
UNFREEZE_VIDEO_TOP = 2                  # v1cc Conformer 6 layer, top 2 trainable (ilk 4 donmus)

# Encoders
VIDEO_ENCODER_VARIANT = 'v1cc'          # v1c veya v1cc (best video model)
VIDEO_CHECKPOINT_SLUG = f'lippy-phoneme-{VIDEO_ENCODER_VARIANT}-full-checkpoint'
HF_AUDIO_MODEL = 'openai/whisper-medium'
AUDIO_CHECKPOINT_SLUG = 'lippy-audio-whisper-ctc-v2-checkpoint'  # PLAN A.2 medium

# Audio constraints
SAMPLE_RATE = 16000
MIN_AUDIO_SEC = 0.5
MAX_AUDIO_SEC = 10.0
WHISPER_MEL_HOP = 160
WHISPER_MAX_MEL_FRAMES = 3000
DYN_MEL_MULTIPLE = 2

# Mel cache (medium 80-bin)
MEL_CACHE_SLUG = 'lippy-mel-cache-whisper-medium'

# Output
DATASET_SLUG = 'lippy-av-fusion-v3-checkpoint'
FORCE_SCRATCH = True                    # v3.2: NaN ckpt'sini atla, sifirdan
MAX_RESUME_PER = 0.95

print(f'EPOCHS={EPOCHS}, BS={BS}xACCUM={ACCUM}=eff{BS*ACCUM}')
print(f'Fusion: {N_FUSION_LAYERS}-layer shared Transformer, d={D_MODEL}, MOD_DROP={MODALITY_DROPOUT}')
print(f'Hybrid loss: {CTC_ATTN_ALPHA}*CTC + {1-CTC_ATTN_ALPHA}*Attn (v3.2: CTC oncelikli)')
print(f'LR: video_top={LR_VIDEO_TOP}, audio_top={LR_AUDIO_TOP}, fusion_head={LR_FUSION_HEAD} (v3.2: 1e-4 NaN-safe)')
print(f'GCLIP={GCLIP} (v3.2: 1.0 sikı clip), WARMUP={WARMUP} (v3.2: 2 smooth ramp)')
print(f'Unfreeze: video top {UNFREEZE_VIDEO_TOP}/6 conformer + audio top {UNFREEZE_AUDIO_TOP}/24 whisper')
print(f'Video variant: {VIDEO_ENCODER_VARIANT}, Audio: {HF_AUDIO_MODEL}')
print(f'Mel cache: {MEL_CACHE_SLUG}')
print(f'FORCE_SCRATCH={FORCE_SCRATCH} (v3.2: NaN ckpt atla)')

## 1) Vocab — 31 fonem (audio v2 + video v1cc ile birebir)

In [ ]:
PHONEMES = [
    'SIL', 'SPACE',
    'a','e','i','o','u','\u0131','\u00f6','\u00fc',
    'b','p','m','f','v','d','t','n','l',
    's','z','c','\u00e7','\u015f','j',
    'k','g','\u011f','r','h','y',
]
BLANK_IDX = 0; SPACE_IDX = 1
BOS_IDX = len(PHONEMES)   # 31 (decoder icin <bos> token, vocab+1)
EOS_IDX = len(PHONEMES)   # ayni token <bos>/<eos> ortak
PAD_IDX = BLANK_IDX       # padding icin blank (CTC ile uyumlu)
NUM_PHONEMES = len(PHONEMES)
DECODER_VOCAB = NUM_PHONEMES + 1  # +1 for <bos>/<eos>

PH_TO_IDX = {p:i for i,p in enumerate(PHONEMES)}
IDX_TO_PH = {i:p for i,p in enumerate(PHONEMES)}
CHAR_TO_PH = {ph:ph for ph in PHONEMES if ph not in ('SIL','SPACE')}

def text_to_phonemes(text):
    phs = []
    for ch in text.lower().strip():
        if ch == ' ':
            if not phs or phs[-1] != SPACE_IDX:
                phs.append(SPACE_IDX)
        elif ch in CHAR_TO_PH:
            phs.append(PH_TO_IDX[CHAR_TO_PH[ch]])
    while phs and phs[0]==SPACE_IDX: phs.pop(0)
    while phs and phs[-1]==SPACE_IDX: phs.pop()
    return phs

def phonemes_to_text(idxs):
    if isinstance(idxs, torch.Tensor): idxs = idxs.tolist()
    out = []
    for i in idxs:
        if i==BLANK_IDX or i==BOS_IDX: continue
        out.append(' ' if i==SPACE_IDX else IDX_TO_PH.get(i,''))
    return ''.join(out)

print(f'Vocab: {NUM_PHONEMES} fonem + 1 BOS = {DECODER_VOCAB} (decoder)')

## 2) Video Encoder — v1cc (donmus + ust 2 Conformer layer TRAINABLE)

Ilk 4 conformer layer + ResNet trunk + frontend3D donmus.
Ust 2 conformer layer trainable -> co-adaptation.

In [ ]:
class SiLU(nn.Module):
    def forward(self,x): return x * torch.sigmoid(x)

class BasicBlockSiLU(nn.Module):
    expansion=1
    def __init__(self,ic,oc,stride=1,ds=None):
        super().__init__()
        self.conv1=nn.Conv2d(ic,oc,3,stride=stride,padding=1,bias=False)
        self.bn1=nn.BatchNorm2d(oc)
        self.conv2=nn.Conv2d(oc,oc,3,padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(oc)
        self.act=SiLU(); self.ds=ds
    def forward(self,x):
        identity = x if self.ds is None else self.ds(x)
        return self.act(self.bn2(self.conv2(self.act(self.bn1(self.conv1(x))))) + identity)

class ResNetSiLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1=self._ml(64,64,2,1)
        self.layer2=self._ml(64,128,2,2)
        self.layer3=self._ml(128,256,2,2)
        self.layer4=self._ml(256,512,2,2)
    def _ml(self,ic,oc,n,s):
        ds = nn.Sequential(nn.Conv2d(ic,oc,1,stride=s,bias=False),nn.BatchNorm2d(oc)) if s!=1 or ic!=oc else None
        return nn.Sequential(BasicBlockSiLU(ic,oc,s,ds), *[BasicBlockSiLU(oc,oc) for _ in range(n-1)])
    def forward(self,x): return self.layer4(self.layer3(self.layer2(self.layer1(x))))

# v1cc dim_ff=2048 (v1c 1024 idi)
V_D_MODEL = 512
V_ENC_LAYERS = 6
V_NHEAD = 8
V_DIM_FF = 2048 if VIDEO_ENCODER_VARIANT == 'v1cc' else 1024

class VideoEncoder(nn.Module):
    """v1cc encoder. get_features dondurur (CTC head'i atilir)."""
    def __init__(self):
        super().__init__()
        self.frontend3D = nn.Sequential(
            nn.Conv3d(1,64,(5,7,7),stride=(1,2,2),padding=(2,3,3),bias=False),
            nn.BatchNorm3d(64), SiLU(),
            nn.MaxPool3d((1,3,3),stride=(1,2,2),padding=(0,1,1)),
        )
        self.trunk = ResNetSiLU()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Identity()
        self.conformer = Conformer(input_dim=V_D_MODEL, num_heads=V_NHEAD, ffn_dim=V_DIM_FF,
                                    num_layers=V_ENC_LAYERS, depthwise_conv_kernel_size=31, dropout=0.1)
        self.ctc_fc = nn.Linear(V_D_MODEL, NUM_PHONEMES)   # ckpt load icin var

    def freeze_except_top(self, n_top=2):
        """Frontend + trunk + ilk (6-n) conformer donmus."""
        for p in self.frontend3D.parameters(): p.requires_grad = False
        for p in self.trunk.parameters(): p.requires_grad = False
        for p in self.ctc_fc.parameters(): p.requires_grad = False
        # Conformer layers: ilk (6-n) donmus
        if hasattr(self.conformer, 'conformer_layers'):
            layers = self.conformer.conformer_layers
        else:
            layers = list(self.conformer.children())
        n_freeze = max(0, len(layers) - n_top)
        for i, layer in enumerate(layers):
            if i < n_freeze:
                for p in layer.parameters(): p.requires_grad = False
        n_train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  Video: ilk {n_freeze}/{len(layers)} conformer DONMUS, ust {n_top} TRAINABLE ({n_train/1e6:.1f}M)')

    def get_features(self, x, il=None):
        B,T,H,W = x.shape
        x = x.unsqueeze(1)
        x = self.frontend3D(x)
        B2,C,To,H2,W2 = x.shape
        x = x.permute(0,2,1,3,4).reshape(B2*To, C, H2, W2)
        x = self.avg_pool(self.trunk(x)).view(B2, To, -1)
        x = self.proj(x)
        if il is None:
            il = torch.full((B2,), To, dtype=torch.long, device=x.device)
        x, lengths = self.conformer(x, lengths=il)
        return x, lengths  # (B, T_v, 512)

## 3) Audio Encoder — Whisper-medium (donmus + ust 4 layer TRAINABLE)

Conv1, conv2, pos embed + ilk 20 transformer layer donmus.
Ust 4 layer (idx 20-23) trainable -> co-adaptation.

In [ ]:
class AudioEncoderWhisper(nn.Module):
    def __init__(self, model_name=HF_AUDIO_MODEL):
        super().__init__()
        full = WhisperModel.from_pretrained(model_name, torch_dtype=torch.float32)
        self.encoder = full.encoder
        self.encoder_dropout_p = 0.1   # FIX v2: encoder.dropout=0.0 idi (no-op)
        del full; torch.cuda.empty_cache()
        self.hidden = self.encoder.config.d_model   # medium=1024
        self.n_layers = len(self.encoder.layers)    # medium=24
        self.ctc_fc = nn.Linear(self.hidden, NUM_PHONEMES)  # PLAN A.2 ckpt'ten load icin

    def freeze_except_top(self, n_top=4):
        """Conv + pos + ilk (24-n) transformer donmus. CTC head donmus (zaten kullanmiyoruz)."""
        for p in self.encoder.conv1.parameters(): p.requires_grad = False
        for p in self.encoder.conv2.parameters(): p.requires_grad = False
        self.encoder.embed_positions.requires_grad = False
        for p in self.ctc_fc.parameters(): p.requires_grad = False
        n_freeze = max(0, self.n_layers - n_top)
        for i, layer in enumerate(self.encoder.layers):
            if i < n_freeze:
                for p in layer.parameters(): p.requires_grad = False
        n_train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  Audio: ilk {n_freeze}/{self.n_layers} whisper layer DONMUS, ust {n_top} TRAINABLE ({n_train/1e6:.1f}M)')

    def get_features(self, input_features):
        """Dynamic length forward (pos embed dilimle)."""
        x = F.gelu(self.encoder.conv1(input_features))
        x = F.gelu(self.encoder.conv2(x))
        x = x.permute(0, 2, 1)  # (B, T_out, hidden)
        T_out = x.shape[1]
        embed_pos = self.encoder.embed_positions.weight[:T_out]
        x = x + embed_pos
        x = F.dropout(x, p=self.encoder_dropout_p, training=self.training)
        for layer in self.encoder.layers:
            x = layer(x, None, None)[0]
        x = self.encoder.layer_norm(x)
        return x   # (B, T_out, hidden=1024)

## 4) Symmetric Fusion + Hybrid CTC/Attention Head + AVFusionV3 model

**Fusion:** concat (V + downsampled A) -> Linear(2D->D) -> 8-layer Transformer + modality dropout.
**Head:** CTC (linear) + Attention seq2seq decoder (4 layer).

In [ ]:
class SymmetricFusion(nn.Module):
    """Concat + temporal align + shared Transformer.
    Audio 50Hz -> downsample 2x -> 25Hz (video framerate ile esle).
    Modality dropout: train'de %25 V mask, %25 A mask, %50 normal.
    """
    def __init__(self, d=D_MODEL, audio_dim=1024, video_dim=V_D_MODEL,
                 n_layers=N_FUSION_LAYERS, nhead=NHEAD,
                 dropout=FUSION_DROPOUT, modality_dropout=MODALITY_DROPOUT):
        super().__init__()
        self.video_proj = nn.Linear(video_dim, d)
        self.audio_proj = nn.Linear(audio_dim, d)
        # Audio temporal downsample (50Hz -> 25Hz)
        self.audio_downsample = nn.Conv1d(d, d, kernel_size=2, stride=2)
        # Concat -> proj
        self.combine = nn.Linear(2 * d, d)
        # Shared Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=nhead, dim_feedforward=2048, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.shared = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.modality_dropout = modality_dropout
        self.dropout = nn.Dropout(dropout)

    def _align_audio(self, a_feat, target_T):
        """Audio'yu hedef temporal uzunluga uyarla (downsample veya interp)."""
        # a_feat: (B, T_a, D)
        a = a_feat.transpose(1, 2)  # (B, D, T_a)
        a = self.audio_downsample(a)  # (B, D, T_a/2)
        # T_a/2 ile target_T arasinda hala fark olabilir, linear interp
        if a.shape[-1] != target_T:
            a = F.interpolate(a, size=target_T, mode='linear', align_corners=False)
        return a.transpose(1, 2)  # (B, target_T, D)

    def forward(self, v_feat, a_feat, v_mask=None):
        """v_feat (B, T_v, V_D), a_feat (B, T_a, A_D), v_mask (B, T_v) True=padding."""
        v = self.video_proj(v_feat)        # (B, T_v, D)
        a_proj = self.audio_proj(a_feat)   # (B, T_a, D)
        a_aligned = self._align_audio(a_proj, v.shape[1])  # (B, T_v, D)

        # Modality dropout (train only)
        if self.training and self.modality_dropout > 0:
            r = torch.rand(v.shape[0], device=v.device)
            # %25 V mask, %25 A mask, %50 normal
            v_mask_mod = (r < self.modality_dropout / 2).view(-1, 1, 1).float()
            a_mask_mod = ((r >= self.modality_dropout / 2) & (r < self.modality_dropout)).view(-1, 1, 1).float()
            v = v * (1 - v_mask_mod)
            a_aligned = a_aligned * (1 - a_mask_mod)

        fused = torch.cat([v, a_aligned], dim=-1)  # (B, T_v, 2D)
        fused = self.combine(fused)                # (B, T_v, D)
        fused = self.dropout(fused)
        out = self.shared(fused, src_key_padding_mask=v_mask)  # (B, T_v, D)
        return out


class HybridHead(nn.Module):
    """CTC + Attention seq2seq decoder."""
    def __init__(self, d=D_MODEL, vocab=NUM_PHONEMES, decoder_vocab=DECODER_VOCAB,
                 n_decoder_layers=N_DECODER_LAYERS, nhead=NHEAD, dropout=FUSION_DROPOUT):
        super().__init__()
        # CTC branch
        self.ctc_proj = nn.Linear(d, vocab)
        # Attention decoder branch
        self.tgt_embed = nn.Embedding(decoder_vocab, d, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(512, d)  # max 512 token tgt len
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d, nhead=nhead, dim_feedforward=2048, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=n_decoder_layers)
        self.attn_out = nn.Linear(d, decoder_vocab)

    def forward_ctc(self, encoder_out):
        """(B, T, D) -> (T, B, vocab) log_probs for CTC."""
        return self.ctc_proj(encoder_out).log_softmax(-1).permute(1, 0, 2)

    def forward_attn(self, encoder_out, tgt_in, memory_key_padding_mask=None):
        """Teacher forcing decoder forward.
        encoder_out: (B, T_v, D), tgt_in: (B, L) with <bos> prefix
        Returns: (B, L, decoder_vocab) logits
        """
        B, L = tgt_in.shape
        positions = torch.arange(L, device=tgt_in.device).unsqueeze(0).expand(B, -1)
        tgt = self.tgt_embed(tgt_in) + self.pos_embed(positions)
        # Causal mask
        causal = torch.triu(torch.ones(L, L, device=tgt_in.device), diagonal=1).bool()
        out = self.decoder(tgt, encoder_out, tgt_mask=causal,
                            memory_key_padding_mask=memory_key_padding_mask,
                            tgt_key_padding_mask=(tgt_in == PAD_IDX))
        return self.attn_out(out)  # (B, L, decoder_vocab)


class AVFusionV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.video_encoder = VideoEncoder()
        self.audio_encoder = AudioEncoderWhisper()
        self.fusion = SymmetricFusion(
            d=D_MODEL,
            audio_dim=self.audio_encoder.hidden,
            video_dim=V_D_MODEL,
        )
        self.head = HybridHead(d=D_MODEL)

    def freeze_encoders_partial(self):
        """v3: ust N layer trainable, alt donmus."""
        self.video_encoder.freeze_except_top(UNFREEZE_VIDEO_TOP)
        self.audio_encoder.freeze_except_top(UNFREEZE_AUDIO_TOP)
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f'AVFusionV3 trainable: {trainable/1e6:.1f}M / total {total/1e6:.1f}M ({100*trainable/total:.1f}%)')

    def forward(self, video_input, audio_input, video_il=None, tgt_in=None):
        """
        video_input: (B, T_v_frames, H, W)
        audio_input: (B, 80, T_mel) -- Whisper mel features
        video_il: (B,) gercek video frame sayisi
        tgt_in: (B, L) decoder input with <bos> prefix (training only, teacher forcing)
        """
        v_feat, v_lens = self.video_encoder.get_features(video_input, il=video_il)
        a_feat = self.audio_encoder.get_features(audio_input)

        # v_mask (True=padding)
        B, T_v, _ = v_feat.shape
        v_mask = torch.arange(T_v, device=v_feat.device).unsqueeze(0) >= v_lens.unsqueeze(1)

        # Fusion
        fused = self.fusion(v_feat, a_feat, v_mask=v_mask)  # (B, T_v, D)

        # CTC log_probs
        ctc_log_probs = self.head.forward_ctc(fused)  # (T_v, B, vocab)

        # Attention logits (sadece training/teacher forcing)
        attn_logits = None
        if tgt_in is not None:
            attn_logits = self.head.forward_attn(fused, tgt_in, memory_key_padding_mask=v_mask)

        return ctc_log_probs, attn_logits, v_lens

## 5) Ckpt loading — v1cc video + medium PLAN A.2 audio (state_dict transfer)

Sadece encoder agirliklari yuklenir. CTC head'leri atilir (v3'te yeni hybrid head var).

In [ ]:
model = AVFusionV3().to(DEVICE)

# Video encoder pretrained yukle
print(f'\nVideo encoder pretrained: {VIDEO_ENCODER_VARIANT}')
video_cands = glob.glob(f'/kaggle/input/**/{VIDEO_CHECKPOINT_SLUG}/**/best_model.pt', recursive=True)
if not video_cands:
    video_cands = glob.glob('/kaggle/input/**/best_model.pt', recursive=True)
    video_cands = [p for p in video_cands if VIDEO_ENCODER_VARIANT in p.lower()]
if video_cands:
    print(f'  Yukleniyor: {video_cands[0]}')
    state = torch.load(video_cands[0], map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    missing, unexpected = model.video_encoder.load_state_dict(state, strict=False)
    matched = len(state) - len(unexpected)
    print(f'  Video matched: {matched}/{len(state)} keys, unexpected={len(unexpected)}')
else:
    print('  UYARI: Video ckpt YOK, scratch baslayacak')

# Audio encoder pretrained yukle (PLAN A.2 medium)
print(f'\nAudio encoder pretrained: {HF_AUDIO_MODEL} (PLAN A.2 medium ckpt)')
audio_cands = glob.glob(f'/kaggle/input/**/{AUDIO_CHECKPOINT_SLUG}/**/best_model.pt', recursive=True)
if audio_cands:
    print(f'  Yukleniyor: {audio_cands[0]}')
    state = torch.load(audio_cands[0], map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    # Whisper hidden boyut kontrolu (medium=1024, large-v3=1280)
    sample_k = next((k for k in state.keys() if 'encoder.layers.0.fc1.weight' in k), None)
    if sample_k:
        ckpt_hidden = state[sample_k].shape[1]
        if ckpt_hidden != model.audio_encoder.hidden:
            raise AssertionError(
                f'Audio ckpt hidden={ckpt_hidden} != model hidden={model.audio_encoder.hidden}. '
                f'HF_AUDIO_MODEL={HF_AUDIO_MODEL!r} (medium=1024 bekleniyor)'
            )
    missing, unexpected = model.audio_encoder.load_state_dict(state, strict=False)
    matched = len(state) - len(unexpected)
    print(f'  Audio matched: {matched}/{len(state)} keys, unexpected={len(unexpected)}')
else:
    print('  UYARI: Audio ckpt YOK, HF pretrained kullanilacak')

# Encoder'lari kismi dondur
model.freeze_encoders_partial()

## 6) Joint Dataset — ROI + Audio + Text (mel cache destekli)

In [ ]:
ROIS_CANDS = glob.glob('/kaggle/input/**/rois_mp4', recursive=True)
ROIS_DIR = ROIS_CANDS[0] if ROIS_CANDS else '/kaggle/input/lippy-test/dataset/rois_mp4'
AUDIO_CANDS = (glob.glob('/kaggle/input/**/lippy-audio-segments/**/audios', recursive=True)
                + glob.glob('/kaggle/input/**/audios', recursive=True))
AUDIO_DIR = AUDIO_CANDS[0] if AUDIO_CANDS else '/kaggle/input/lippy-audio-segments/audios'
TEXT_CANDS = glob.glob('/kaggle/input/**/dataset/texts', recursive=True)
TEXT_DIR = TEXT_CANDS[0] if TEXT_CANDS else '/kaggle/input/lippy-test/dataset/texts'
SPLIT_CANDS = glob.glob('/kaggle/input/**/dataset/splits', recursive=True)
SPLIT_DIR = SPLIT_CANDS[0] if SPLIT_CANDS else '/kaggle/input/lippy-test/dataset/splits'
print(f'ROIS: {ROIS_DIR}')
print(f'AUDIO: {AUDIO_DIR}')
print(f'TEXT: {TEXT_DIR}')
print(f'SPLIT: {SPLIT_DIR}')

# Mel cache (fast-path)
def _find_mel_cache():
    user = os.environ.get('KAGGLE_USERNAME', 'brokyewing')
    for cand in [f'/kaggle/input/datasets/{user}/{MEL_CACHE_SLUG}',
                  f'/kaggle/input/{MEL_CACHE_SLUG}']:
        if os.path.isdir(cand):
            for e in os.scandir(cand):
                if e.name.endswith('.pt'): return cand
    return None
MEL_CACHE_DIR = _find_mel_cache()
USE_MEL_CACHE = MEL_CACHE_DIR is not None
if USE_MEL_CACHE:
    cn = sum(1 for f in os.scandir(MEL_CACHE_DIR) if f.name.endswith('.pt'))
    print(f'Mel cache AKTIF: {MEL_CACHE_DIR} ({cn:,} dosya)')
else:
    print(f'Mel cache YOK: on-the-fly Whisper FE (yavas)')

whisper_fe = WhisperFeatureExtractor.from_pretrained(HF_AUDIO_MODEL)
N_MELS = whisper_fe.feature_size

MEAN_STD = {'v1c': (0.421, 0.165), 'v1cc': (0.495, 0.200)}
MEAN, STD = MEAN_STD[VIDEO_ENCODER_VARIANT]
CROP = 88
print(f'Video normalize: mean={MEAN}, std={STD} ({VIDEO_ENCODER_VARIANT})')

# FIX v3.1: Dataloader stability config
# - Retry budget 53K -> 100 (hizli fail, worker hang/leak'i ortaya cikar)
# - num_workers 4 -> 2 + persistent_workers False (memory leak fix)
# - CTC safety 2x -> 1x strict (CTC gercek requirement, daha az sample skip)
RETRY_BUDGET = 100
DATALOADER_NUM_WORKERS = 2

def read_mp4(p):
    cap = cv2.VideoCapture(p)
    frames = []
    while True:
        r, fr = cap.read()
        if not r: break
        frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
    cap.release()
    return np.array(frames, dtype=np.uint8) if frames else None

def load_mel_from_cache(s):
    if not USE_MEL_CACHE: return None
    p = f'{MEL_CACHE_DIR}/{s}.pt'
    if not os.path.exists(p): return None
    try: return torch.load(p, weights_only=True).float()
    except: return None

def load_mel_from_wav(s):
    p = os.path.join(AUDIO_DIR, f'{s}.wav')
    if not os.path.exists(p): return None
    wav, sr = torchaudio.load(p)
    if sr != SAMPLE_RATE: wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
    wav = wav.squeeze(0).float().numpy()
    if len(wav) < SAMPLE_RATE * MIN_AUDIO_SEC: return None
    if len(wav) > SAMPLE_RATE * MAX_AUDIO_SEC:
        t = int(SAMPLE_RATE * MAX_AUDIO_SEC)
        s_ = (len(wav) - t) // 2
        wav = wav[s_:s_+t]
    f = whisper_fe(wav, sampling_rate=SAMPLE_RATE, return_tensors='pt',
                    padding=False, return_attention_mask=False).input_features
    return f.squeeze(0).float()

class JointDataset(Dataset):
    def __init__(self, split):
        self.train = split == 'train'
        with open(os.path.join(SPLIT_DIR, f'{split}.txt'), encoding='utf-8') as f:
            all_ids = [l.strip() for l in f if l.strip()]
        self.ids = []
        for s in all_ids:
            rp = os.path.join(ROIS_DIR, f'{s}.mp4')
            tp = os.path.join(TEXT_DIR, f'{s}.txt')
            # Audio: cache veya wav
            has_audio = (USE_MEL_CACHE and os.path.exists(f'{MEL_CACHE_DIR}/{s}.pt')) \
                         or os.path.exists(os.path.join(AUDIO_DIR, f'{s}.wav'))
            if os.path.exists(rp) and os.path.exists(tp) and has_audio:
                self.ids.append(s)
        print(f'  [{split}] {len(self.ids)}/{len(all_ids)}')

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        # FIX v3.1: retry budget 100 (53K -> 100 -- hizli fail, worker leak/hang gorulur)
        for off in range(min(RETRY_BUDGET, len(self.ids))):
            s = self.ids[(i + off) % len(self.ids)]
            try:
                roi = read_mp4(os.path.join(ROIS_DIR, f'{s}.mp4'))
                if roi is None or len(roi) < 5: continue
                roi = roi.astype(np.float32) / 255.0
                T, H, W = roi.shape
                if self.train:
                    t = random.randint(0, max(0, H-CROP)); l = random.randint(0, max(0, W-CROP))
                    if random.random() > 0.5: roi = roi[:, :, ::-1].copy()
                else:
                    t = (H-CROP)//2; l = (W-CROP)//2
                roi = roi[:, t:t+CROP, l:l+CROP]
                roi_t = torch.from_numpy(((roi - MEAN) / STD).copy()).float()

                # Audio (mel cache fast path)
                mel = load_mel_from_cache(s)
                if mel is None:
                    mel = load_mel_from_wav(s)
                if mel is None: continue
                T_mel = mel.shape[1]
                if T_mel > WHISPER_MAX_MEL_FRAMES:
                    mel = mel[:, :WHISPER_MAX_MEL_FRAMES]
                    T_mel = WHISPER_MAX_MEL_FRAMES
                if T_mel % 2 != 0:
                    mel = mel[:, :-1]; T_mel -= 1

                # Text
                with open(os.path.join(TEXT_DIR, f'{s}.txt'), encoding='utf-8') as f:
                    txt = f.read().strip()
                lab = torch.tensor(text_to_phonemes(txt), dtype=torch.long)
                if len(lab) == 0: continue

                # FIX v3.1: CTC safety STRICT (1x) instead of 2x.
                # CTC actual minimum: input_length >= label_length.
                # 2x was just safety margin -- skip ediliyordu fazla sample.
                if roi_t.shape[0] < len(lab): continue

                return roi_t, mel, lab, T_mel
            except Exception:
                continue
        # Eger 100 deneme sonra hala bulamadiysak, raise et (worker hata loglansin)
        raise RuntimeError(f'Sample yuklenemedi after {RETRY_BUDGET} retries (worker leak/hang ihtimal)')

def collate(batch):
    rois, mels, labs, T_mels = zip(*batch)
    # Video padding (batch-max)
    il_v = torch.tensor([r.shape[0] for r in rois], dtype=torch.long)
    pad_v = torch.zeros(len(rois), il_v.max().item(), CROP, CROP)
    for i, r in enumerate(rois): pad_v[i, :r.shape[0]] = r
    # Audio padding (batch-max mel)
    T_mel_b = max(T_mels)
    if T_mel_b % DYN_MEL_MULTIPLE != 0:
        T_mel_b += DYN_MEL_MULTIPLE - (T_mel_b % DYN_MEL_MULTIPLE)
    T_mel_b = min(T_mel_b, WHISPER_MAX_MEL_FRAMES)
    pad_a = torch.zeros(len(mels), N_MELS, T_mel_b)
    for i, m in enumerate(mels):
        L = min(m.shape[1], T_mel_b)
        pad_a[i, :, :L] = m[:, :L]
    # Labels (CTC ile cat, attention ile padded)
    tl = torch.tensor([l.shape[0] for l in labs], dtype=torch.long)
    labs_cat = torch.cat(labs)
    # Decoder input: <bos> + label, padded
    max_L = tl.max().item() + 1  # +1 for <bos>
    tgt_in = torch.full((len(labs), max_L), PAD_IDX, dtype=torch.long)
    tgt_out = torch.full((len(labs), max_L), PAD_IDX, dtype=torch.long)
    for i, l in enumerate(labs):
        L = l.shape[0]
        tgt_in[i, 0] = BOS_IDX
        tgt_in[i, 1:1+L] = l
        tgt_out[i, :L] = l
        tgt_out[i, L] = EOS_IDX
    return pad_v, pad_a, labs_cat, il_v, tl, tgt_in, tgt_out

# FIX v3.1: num_workers 4 -> 2 + persistent_workers False (worker memory leak fix)
trl = DataLoader(JointDataset('train'), batch_size=BS, shuffle=True, collate_fn=collate,
                  num_workers=DATALOADER_NUM_WORKERS, pin_memory=True, drop_last=True,
                  persistent_workers=False, prefetch_factor=2)
_val_full = JointDataset('val')
_val_subset = torch.utils.data.Subset(_val_full, list(range(0, len(_val_full), 3)))
vll = DataLoader(_val_subset, batch_size=BS, shuffle=False, collate_fn=collate,
                  num_workers=DATALOADER_NUM_WORKERS, pin_memory=True,
                  persistent_workers=False, prefetch_factor=2)
print(f'Train: {len(trl)} batch | Val (subset 1/3): {len(vll)} batch')
print(f'DataLoader: num_workers={DATALOADER_NUM_WORKERS}, persistent_workers=False (leak fix)')
print(f'CTC safety: strict (1x label, was 2x) -- daha az sample skip')
print(f'Retry budget: {RETRY_BUDGET} (was {53000}+) -- hizli fail worker hang gorulur')

## 7) Training setup — 3-grup LR + hybrid loss + resume + upload

In [ ]:
MDIR = '/kaggle/working/ckpt_av_fusion_v3'
UDIR = '/kaggle/working/upload_av_fusion_v3'
os.makedirs(MDIR, exist_ok=True); os.makedirs(UDIR, exist_ok=True)
CP = f'{MDIR}/checkpoint.pt'; BP = f'{MDIR}/best_model.pt'

KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', 'UNKNOWN')
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    KAGGLE_USER = _us.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY'] = _us.get_secret('KAGGLE_KEY')
except: pass

DATASET_ID = f'{KAGGLE_USER}/{DATASET_SLUG}'
with open(f'{UDIR}/dataset-metadata.json', 'w') as f:
    json.dump({'title':'LipPY AV-Fusion v3 (co-train + symmetric + hybrid)','id':DATASET_ID,
                'licenses':[{'name':'CC0-1.0'}]}, f)

def greedy_ctc(lp):
    ids = lp.argmax(-1).tolist()
    d=[]; p=-1
    for i in ids:
        if i != p and i != BLANK_IDX: d.append(i)
        p = i
    return d

def edit_dist(a, b):
    n,m=len(a),len(b); dp=list(range(m+1))
    for ai in a:
        r=[dp[0]+1]
        for j,bj in enumerate(b):
            r.append(min(r[-1]+1, dp[j+1]+1, dp[j]+(0 if ai==bj else 1)))
        dp=r
    return dp[-1]

def per_calc(preds, refs):
    td=tl=0
    for p,r in zip(preds, refs):
        td+=edit_dist(p,r); tl+=max(len(r),1)
    return td/max(tl,1)

def metrics(preds, refs):
    per_all = per_calc(preds, refs)
    preds_ph = [[t for t in p if t != SPACE_IDX] for p in preds]
    refs_ph = [[t for t in r if t != SPACE_IDX] for r in refs]
    per_phoneme = per_calc(preds_ph, refs_ph)
    tp=fp=fn=0
    for p,r in zip(preds, refs):
        ps=sum(1 for t in p if t==SPACE_IDX); rs=sum(1 for t in r if t==SPACE_IDX)
        tp+=min(ps,rs); fp+=max(0,ps-rs); fn+=max(0,rs-ps)
    prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
    f1=2*prec*rec/max(prec+rec,1e-6)
    return {'per_all':per_all, 'per_phoneme':per_phoneme, 'space_f1':f1}

def cosine_warmup(ep, warm, total, min_r=0.05):
    if ep < warm: return max(0.1, (ep+1)/warm)
    p = (ep-warm) / max(1, total-warm)
    return max(min_r, 0.5*(1+math.cos(math.pi*p)))

# 3-grup optimizer (LR layer-specific)
video_top_params = [p for p in model.video_encoder.parameters() if p.requires_grad]
audio_top_params = [p for p in model.audio_encoder.parameters() if p.requires_grad]
fusion_head_params = list(model.fusion.parameters()) + list(model.head.parameters())

opt = AdamW([
    {'params': video_top_params, 'lr': LR_VIDEO_TOP, 'weight_decay': 0.01},
    {'params': audio_top_params, 'lr': LR_AUDIO_TOP, 'weight_decay': 0.01},
    {'params': fusion_head_params, 'lr': LR_FUSION_HEAD, 'weight_decay': 0.01},
], betas=(0.9, 0.98))

sched = LambdaLR(opt, [lambda e: cosine_warmup(e, WARMUP, EPOCHS)] * 3)
ctc_crit = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
ce_crit = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

print(f'Trainable params: video_top={sum(p.numel() for p in video_top_params)/1e6:.1f}M, '
      f'audio_top={sum(p.numel() for p in audio_top_params)/1e6:.1f}M, '
      f'fusion_head={sum(p.numel() for p in fusion_head_params)/1e6:.1f}M')

# Resume
start_ep = 1; best_per = 1.0
if not FORCE_SCRATCH:
    ckpts = ([CP] if os.path.exists(CP) else []) + glob.glob(
        f'/kaggle/input/**/{DATASET_SLUG}/**/checkpoint.pt', recursive=True)
    ckpts = sorted(set(ckpts), key=lambda p: -os.path.getmtime(p) if os.path.exists(p) else 0)
    if ckpts:
        try:
            ck = torch.load(ckpts[0], map_location=DEVICE, weights_only=False)
            cp = ck.get('best_per', 1.0)
            if cp <= MAX_RESUME_PER:
                model.load_state_dict(ck['model_state_dict'], strict=False)
                start_ep = ck.get('epoch', 0) + 1
                best_per = cp
                try:
                    opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
                except: pass
                if 'scaler' in ck: scaler.load_state_dict(ck['scaler'])
                print(f'Resume OK: ep {ck.get("epoch",0)} best_per={best_per:.1%}')
        except Exception as e:
            print(f'Resume HATA: {e}')

def upload(ep):
    for f in glob.glob(f'{MDIR}/*.pt'):
        shutil.copy(f, f'{UDIR}/{os.path.basename(f)}')
    r = subprocess.run(['kaggle','datasets','version','-p',UDIR,
                         '-m',f'av-fusion-v3 ep{ep}','--dir-mode','zip'],
                       capture_output=True, text=True, timeout=1800)
    if r.returncode == 0: print(f'[upload ep{ep}] OK')
    elif 'not found' in (r.stderr or '').lower() or '404' in (r.stderr or '').lower():
        cr = subprocess.run(['kaggle','datasets','create','-p',UDIR,'--dir-mode','zip'],
                              capture_output=True, text=True, timeout=1800)
        print(f'[upload ep{ep}] CREATE rc={cr.returncode}')
    else: print(f'[upload ep{ep}] FAIL rc={r.returncode}')

## 8) Training loop (hybrid loss + OOM-safe + session budget)

In [ ]:
print(f'\n--- AV-Fusion v3 Egitim (start_ep={start_ep}) ---\n')
history = []
SESSION_EPOCH_LIMIT = EPOCHS
SLOW_THRESHOLD = 2.5 * 3600   # ep > 2.5h ise session budget devreye

for epoch in range(start_ep, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tl_s = tc = nan = oom = 0
    opt.zero_grad(set_to_none=True)

    for step, (vid, aud, labs_cat, il_v, tl, tgt_in, tgt_out) in enumerate(trl):
        vid = vid.to(DEVICE); aud = aud.to(DEVICE); labs_cat = labs_cat.to(DEVICE)
        il_v = il_v.to(DEVICE); tgt_in = tgt_in.to(DEVICE); tgt_out = tgt_out.to(DEVICE)

        try:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                ctc_log_probs, attn_logits, v_lens = model(vid, aud, video_il=il_v, tgt_in=tgt_in)
            T_out = ctc_log_probs.shape[0]
            il_clamp = v_lens.clamp(max=T_out)
            # CTC loss
            ctc_loss = ctc_crit(ctc_log_probs.float(), labs_cat, il_clamp.cpu(), tl)
            # Attention CE loss
            attn_loss = ce_crit(attn_logits.reshape(-1, DECODER_VOCAB), tgt_out.reshape(-1))
            # Hybrid
            loss = (CTC_ATTN_ALPHA * ctc_loss + (1 - CTC_ATTN_ALPHA) * attn_loss) / ACCUM
            if torch.isnan(loss) or torch.isinf(loss):
                nan += 1
                if (step + 1) % ACCUM == 0: opt.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
        except RuntimeError as e:
            if 'out of memory' in str(e):
                oom += 1; torch.cuda.empty_cache()
                opt.zero_grad(set_to_none=True); continue
            raise

        if (step + 1) % ACCUM == 0 or (step + 1) == len(trl):
            try:
                scaler.unscale_(opt)
                gn = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GCLIP)
                if torch.isfinite(gn):
                    scaler.step(opt); tl_s += loss.item() * ACCUM; tc += 1
                else: nan += 1
                scaler.update()
            except RuntimeError as e:
                if 'out of memory' in str(e):
                    oom += 1; torch.cuda.empty_cache()
                else: raise
            opt.zero_grad(set_to_none=True)

    # Val (CTC-only inference, attention decoder atlandi -- hizli)
    model.eval()
    preds = []; refs = []; vl_s = vc = 0
    with torch.no_grad():
        for vid, aud, labs_cat, il_v, tl, tgt_in, tgt_out in vll:
            vid = vid.to(DEVICE); aud = aud.to(DEVICE); labs_cat = labs_cat.to(DEVICE)
            il_v = il_v.to(DEVICE)
            try:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    ctc_log_probs, _, v_lens = model(vid, aud, video_il=il_v, tgt_in=None)
            except RuntimeError as e:
                if 'out of memory' in str(e): torch.cuda.empty_cache(); continue
                raise
            T_out = ctc_log_probs.shape[0]
            il_clamp = v_lens.clamp(max=T_out)
            try:
                vl = ctc_crit(ctc_log_probs.float(), labs_cat, il_clamp.cpu(), tl)
                if not (torch.isnan(vl) or torch.isinf(vl)):
                    vl_s += vl.item(); vc += 1
            except: pass
            for i in range(vid.shape[0]):
                preds.append(greedy_ctc(ctc_log_probs[:il_clamp[i], i]))
            off = 0
            labs_cpu = labs_cat.cpu()
            for i in range(vid.shape[0]):
                refs.append(labs_cpu[off:off+tl[i]].tolist())
                off += tl[i].item()

    train_loss = tl_s / max(tc, 1)
    val_loss = vl_s / max(vc, 1)
    m = metrics(preds, refs)

    if m['per_phoneme'] < best_per:
        best_per = m['per_phoneme']
        torch.save(model.state_dict(), BP)
        print(f'  [BEST] PER-ph {best_per:.1%}')

    sched.step()
    ns = f' NaN:{nan}' if nan else ''
    os_ = f' OOM:{oom}' if oom else ''
    ep_sec = int(time.time() - t0)
    print(f'Ep {epoch:2d}/{EPOCHS} | TrL:{train_loss:.3f} | VaL:{val_loss:.3f} | '
          f'gap:{(train_loss-val_loss):+.2f} | PER-all:{m["per_all"]:.1%} ph:{m["per_phoneme"]:.1%} | '
          f'SP F1:{m["space_f1"]:.2f} | {ep_sec}s{ns}{os_}')

    if epoch % 3 == 0 or epoch == 1 or epoch == EPOCHS:
        for i in range(min(2, len(preds))):
            print(f'  H: "{phonemes_to_text(refs[i])[:80]}"')
            print(f'  T: "{phonemes_to_text(preds[i])[:80]}"')

    history.append({'epoch':epoch, 'per_all':m['per_all'], 'per_phoneme':m['per_phoneme'],
                     'space_f1':m['space_f1'], 'val_loss':val_loss})

    torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                'opt':opt.state_dict(), 'sched':sched.state_dict(),
                'scaler':scaler.state_dict(), 'best_per':best_per}, CP)

    if epoch == start_ep and ep_sec > SLOW_THRESHOLD:
        SESSION_EPOCH_LIMIT = start_ep + 1
        print(f'  [BUDGET] ep{epoch} {ep_sec}s -> ep{SESSION_EPOCH_LIMIT} sonra durdur')

    if epoch % UPLOAD_EVERY == 0: upload(epoch)

    if epoch >= SESSION_EPOCH_LIMIT:
        upload(epoch)
        print(f'\n[BUDGET] Session limit -> cikiliyor. Resume otomatik.')
        break

if epoch >= EPOCHS: upload(EPOCHS)

print('\n' + '=' * 70)
print('AV-FUSION v3 SONUC')
print('=' * 70)
for h in history:
    print(f'ep {h["epoch"]:>2}: PER-all {h["per_all"]:.1%} | PER-ph {h["per_phoneme"]:.1%} | VaL {h["val_loss"]:.3f}')
print(f'BEST: {best_per:.1%}')
print(f'\nKarsilastir:')
print(f'  v1c video-only:     PER 50.3%')
print(f'  Audio medium v2:    PER ~30%')
print(f'  AV-Fusion v2 (eski): PER ~33%')
print(f'  AV-Fusion v3 (BU):   PER {best_per:.1%}')